# Mean Teacher on face anti-spoofing: experiment notebook (Part 2)

This notebook runs the experiments from the Part 2 report, one step at a time:

| Experiment | What it is | Step |
|---|---|---|
| **E0** | SVHN sanity check with the paper's ConvNet (supervised, Pi model, Mean Teacher) | 8 |
| **E1** | Supervised-only baseline on CeFA Protocol 2 | 10-11 |
| **E2** | Pi model (optional) | 11 |
| **E3** | Mean Teacher (EMA teacher), MobileNetV3-Small student | 10-11 |
| **E4** | Frozen ResNet-50 teacher | 10-11 |
| **E5** | U-visible vs U-hidden (does the other attack instrument help when unlabeled?) | 12 |

**How to use it**

1. Use a Colab (or Kaggle) GPU runtime. Run the cells in order.
2. Leave `SYNTHETIC = True` in Step 0 for the *first* pass. It builds tiny fake data and runs every step in a few minutes, so you find code problems before spending GPU hours. The synthetic numbers mean nothing.
3. Then set `SYNTHETIC = False`, fill in `build_manifest()` (Step 2) for your CeFA download, and rerun from the top.
4. Each run writes one JSON file into `results/`. Finished runs are skipped when you rerun a cell, so a Colab disconnect costs you at most one run. Point `ROOT` at Google Drive to keep results.

**What I could and could not test.** The data, split, caching, metric and table code was tested offline. The PyTorch cells (models, training, evaluation) could *not* be run where I wrote this, so Step 7 contains smoke tests that check the key formulas in seconds. Do not skip it, and do not start long runs until every check passes.

## Step 0. Configuration

Edit only this cell. Everything the report's Table II calls TBD is set here:
`steps`, `lr`, `lam_c`, `batch_*`, `tau`. Use the timing cell in Step 10 to choose `steps`, and the `lam_c` sweep in Step 10 to choose `lam_c`.

In [ ]:
import os, json, time, copy, math, itertools, warnings
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
SYNTHETIC = False                   # True: fake data to test the pipeline. False: real CeFA / SVHN.
ROOT      = Path(os.environ.get("MT_ROOT", "work"))   # results, SVHN and figures (no CeFA frames here)
# CeFA folder: the frame caches stay inside it because the licence forbids redistribution.
CEFA_DIR  = Path(os.environ.get("CEFA_ROOT", "../../SP26/COMP7003/CeFA_licence/ CASIA-CeFA"))
IMG       = 112                     # face size for CeFA runs (same as the CeFA baseline)
FRAMES_PER_VIDEO = 5
SUBPROTOCOLS = ["2_1", "2_2"]
BUDGETS   = [100, 400]              # number of labeled VIDEOS
SEEDS     = [0, 1, 2]

CFG = dict(
    batch_lab=32, batch_unl=32,     # labeled / unlabeled frames per step
    steps=3000,                     # choose after the timing cell (Step 10)
    eval_every=250,                 # validation curve spacing
    lr=1e-3, weight_decay=1e-4,
    lam_c=100.0,                    # chosen by sweep_lam_c (Step 10): validation EER 3.67 / 2.58 / 2.00 % for 1 / 10 / 100
    ramp_frac=40000 / 180000,       # ramp-up fraction used in the paper (22%)
    alpha_early=0.99, alpha_late=0.999,
    translate_px=7, flip=True, noise_sigma=0.15,
    cons_loss="mse",                # "mse" (paper) or "kl"
    tau=2.0,                        # temperature, used by "kl" only
    teacher_steps=1500,             # fine-tuning steps for the frozen ResNet-50 (E4)
    teacher_lr=1e-4,
)
if SYNTHETIC:                       # tiny settings so the smoke run finishes in minutes
    IMG = 32; BUDGETS = [20, 40]; SEEDS = [0]
    CFG.update(steps=200, eval_every=50, teacher_steps=100, batch_lab=16, batch_unl=16, translate_px=2)

ROOT.mkdir(parents=True, exist_ok=True)
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
print("SYNTHETIC =", SYNTHETIC, "| results ->", RESULTS)

## Step 1. Evaluation code (tested)

APCER, BPCER, ACER, AUC, EER and per-attack-group APCER, with the decision threshold chosen on the **validation** subjects only. A frame or video is accepted as live when its score is at least the threshold. This is the same code as `metrics.py`, which has unit tests with hand-computed cases.

In [ ]:
"""FAS evaluation utilities (ISO/IEC 30107-3 style).

Conventions used everywhere in this file:
  * labels: 1 = live (bona fide), 0 = attack
  * scores: higher = more likely live (e.g. softmax probability of the live class)
  * a sample is ACCEPTED as live when score >= threshold
  * APCER = share of attacks accepted as live
  * BPCER = share of live faces rejected
  * ACER  = (APCER + BPCER) / 2
With one fixed threshold, HTER (FAR/FRR average) is numerically the same as ACER,
so report it once and say so.

The decision threshold must come from the VALIDATION split only.
"""
import numpy as np
from sklearn.metrics import roc_auc_score


def apcer_bpcer(scores, labels, thr):
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    att = scores[labels == 0]
    liv = scores[labels == 1]
    apcer = float(np.mean(att >= thr)) if att.size else float("nan")
    bpcer = float(np.mean(liv < thr)) if liv.size else float("nan")
    return apcer, bpcer


def eer_threshold(scores, labels):
    """Return (threshold, eer) where APCER and BPCER are closest. Use on validation data."""
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    att = np.sort(scores[labels == 0])
    liv = np.sort(scores[labels == 1])
    if att.size == 0 or liv.size == 0:
        raise ValueError("need both live and attack samples to pick a threshold")
    cand = np.unique(scores)
    cand = np.concatenate([cand, [np.inf]])  # threshold that rejects everything
    # count of attacks with score >= thr, and live with score < thr
    apcer = 1.0 - np.searchsorted(att, cand, side="left") / att.size
    bpcer = np.searchsorted(liv, cand, side="left") / liv.size
    gap = np.abs(apcer - bpcer)
    best = np.flatnonzero(gap == gap.min())
    i = best[len(best) // 2]  # middle of the tied thresholds
    thr = cand[i]
    if not np.isfinite(thr):  # degenerate: fall back to the largest finite score
        thr = cand[-2]
    return float(thr), float((apcer[i] + bpcer[i]) / 2)


def evaluate(val_scores, val_labels, test_scores, test_labels, test_groups=None):
    """Pick the threshold on validation, then report test metrics.

    test_groups: optional array of strings, one per test sample (e.g. 'live',
    'print', 'replay', 'papercut', '3d'). Gives per-group APCER for attack groups.
    """
    thr, val_eer = eer_threshold(val_scores, val_labels)
    s = np.asarray(test_scores, dtype=float)
    y = np.asarray(test_labels, dtype=int)
    apcer, bpcer = apcer_bpcer(s, y, thr)
    out = {
        "threshold": thr,
        "val_eer": val_eer,
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": (apcer + bpcer) / 2,
        "auc": float(roc_auc_score(y, s)),
        "test_eer": eer_threshold(s, y)[1],  # informational only: uses test labels
        "error_rate": float(np.mean((s >= thr).astype(int) != y)),
    }
    if test_groups is not None:
        g = np.asarray(test_groups)
        for name in sorted(set(g[y == 0])):
            m = (g == name) & (y == 0)
            out[f"apcer_{name}"] = float(np.mean(s[m] >= thr))
    return out


def top1_error(logits, targets):
    """Multi-class error rate in percent (used for the SVHN sanity check)."""
    logits = np.asarray(logits)
    targets = np.asarray(targets)
    return float(100.0 * np.mean(logits.argmax(axis=1) != targets))

## Step 2. The manifest (real mode only)

Everything downstream needs one table with **one row per RGB video**:

| column | meaning |
|---|---|
| `video_id` | unique string |
| `path` | folder of frame images, or a video file |
| `subject_idx` | subject number **within its ethnicity** (1-500; for 3D masks any integer) |
| `ethnicity` | `"A"`, `"C"`, `"E"`, or `"3D"` for the mask videos |
| `attack` | `"real"`, `"print"`, `"replay"` or `"3d"` |
| `split_2_1`, `split_2_2` | `"train"`, `"val"`, `"test"`, or `""` if the video is not in that sub-protocol's lists |

**How to fill it.** Open the CeFA download and the official Protocol 2 lists, and look at the file names and folder layout *before* writing code. I could not see the data when I wrote this notebook, so `build_manifest()` is a stub. Read the official train, validation and test lists for sub-protocols 2_1 and 2_2 and set the two split columns from them. A video that appears in neither list for a sub-protocol gets `""`.

`check_counts()` then compares your manifest with the counts in the report's Table I. Those counts are my derivation from the CeFA paper (its Table 2 divided by three modalities), so **a mismatch means either your manifest or my derivation is wrong. Find out which before you train.**

In [ ]:
# RGB video counts per split for CeFA Protocol 2, counted in the official video lists (protocol/CASIA_CeFA@p2.*_video_*.txt).
# The first draft derived 1,000 mask videos from the paper's Table 2; the lists hold 1,079 3D-print masks plus
# 44 silicone masks = 1,123 in each test set. Everything else matched the derivation.
EXPECTED = {
    "2_1": {"train": {"real": 600, "print": 1200}, "val": {"real": 300, "print": 600},
            "test":  {"real": 600, "replay": 600, "3d": 1079, "silicone": 44}},
    "2_2": {"train": {"real": 600, "replay": 600}, "val": {"real": 300, "replay": 300},
            "test":  {"real": 600, "print": 1200, "3d": 1079, "silicone": 44}},
}
MANIFEST_COLUMNS = ["video_id", "path", "subject_idx", "ethnicity", "attack", "split_2_1", "split_2_2"]

CEFA_PAI = {"1": "real", "2": "print", "4": "replay"}   # CeFA readme: PAI 1 real, 2 cloth (print), 4 screen (replay)
SPLIT_NAME = {"train": "train", "dev": "val", "test": "test"}

def _parse(path):
    """CASIA_CeFA/Ethnicity/AF/AF-120/1_120_3_2_2/color  or  CASIA_CeFA/Mask/3D-Mask/073/1_073_1_0_2_1_2/color"""
    parts = path.split("/")
    if parts[1] == "Ethnicity":
        eth = {"AF": "A", "CA": "C", "EA": "E"}[parts[2]]
        subj = int(parts[3].split("-")[1])
        return eth, subj, CEFA_PAI[parts[4].split("_")[-1]]
    kind = "3d" if parts[2] == "3D-Mask" else "silicone"
    return "3D", int(parts[3]), kind

def build_manifest():
    """One row per RGB video that appears in any official Protocol 2 list (2_1 or 2_2)."""
    rows = {}
    for sp in SUBPROTOCOLS:
        for split in ["train", "dev", "test"]:
            f = CEFA_DIR / "protocol" / f"CASIA_CeFA@p{sp.replace('_', '.')}_video_{split}.txt"
            for line in f.read_text().splitlines():
                if not line.strip():
                    continue
                path, y = line.split()
                eth, subj, atk = _parse(path)
                assert (atk == "real") == (y == "0"), f"label in list disagrees with the PAI code: {line}"
                r = rows.setdefault(path, dict(video_id=path, path=path, subject_idx=subj, ethnicity=eth, attack=atk,
                                               split_2_1="", split_2_2=""))
                assert r[f"split_{sp}"] == "", f"{path} listed twice in {sp}"
                r[f"split_{sp}"] = SPLIT_NAME[split]
    return pd.DataFrame(list(rows.values()), columns=MANIFEST_COLUMNS)

def check_counts(video_df, expected=EXPECTED):
    """Compare the manifest with the expected counts; print one line per cell. Returns True if all match."""
    ok = True
    for sp, splits in expected.items():
        col = f"split_{sp}"
        for split, want in splits.items():
            got = video_df[video_df[col] == split].attack.value_counts().to_dict()
            for atk in sorted(set(want) | set(got)):
                w, g = want.get(atk, 0), got.get(atk, 0)
                flag = "OK      " if w == g else "MISMATCH"
                ok &= (w == g)
                print(f"{flag} {sp} {split:5s} {atk:7s} expected {w:5d} got {g:5d}")
    return ok

## Step 3. Frames and the cache

`cache_frames()` takes `FRAMES_PER_VIDEO` evenly spaced frames from each video, resizes them to `IMG x IMG`, and stores everything as one `uint8` array (about 1.3 GB for all CeFA RGB videos at 112 px) plus a frame table that repeats the video columns.

**Face crops, not full frames.** The model needs the face region. If your files are 1280x720 frames, crop faces first (use the released crops if the download has them, otherwise a face detector). The function warns when images look like full frames.

In synthetic mode, `make_synthetic()` builds a tiny dataset with the same columns and the same split logic. Its images have simple class patterns (smooth for real, stripes for print, checkerboard for replay, a blob for 3D), so the models can learn something and the unseen-instrument tests behave like the real ones, only easier.

In [ ]:
def _pick(n_avail, k):
    """k evenly spaced indices from range(n_avail)."""
    return np.unique(np.linspace(0, n_avail - 1, k).round().astype(int))

def load_frames_for_video(path, k, size):
    """Return up to k RGB frames (uint8, size x size) from a frame folder or a video file."""
    from PIL import Image
    p = Path(path)
    if p.is_dir():
        files = sorted(f for f in p.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp"))
        out = []
        for i in _pick(len(files), k):
            im = Image.open(files[i]).convert("RGB")
            if max(im.size) > 400:
                warnings.warn(f"{files[i]} is {im.size}: this looks like a full frame, not a face crop")
            out.append(np.asarray(im.resize((size, size), Image.BILINEAR)))
        return out
    import cv2
    cap = cv2.VideoCapture(str(p)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); out = []
    for i in _pick(n, k):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, fr = cap.read()
        if ok:
            if max(fr.shape[:2]) > 400:
                warnings.warn(f"{p} has {fr.shape[:2]} frames: this looks like full frames, not face crops")
            out.append(cv2.resize(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB), (size, size)))
    cap.release()
    return out

def cache_frames(video_df, out_dir=None, size=None, k=None):
    """Build (or load) the frame array and frame table for every video in the manifest."""
    out_dir = Path(out_dir or ROOT); size = size or IMG; k = k or FRAMES_PER_VIDEO
    npy, csv = out_dir / f"frames_{size}_{k}.npy", out_dir / f"frames_{size}_{k}.csv"
    if npy.exists() and csv.exists():
        print("loading cache", npy)
        return np.load(npy), pd.read_csv(csv, keep_default_na=False)
    imgs, rows = [], []
    for n, r in enumerate(video_df.itertuples(index=False)):
        for im in load_frames_for_video(r.path, k, size):
            imgs.append(im)
            rows.append(dict(video_id=r.video_id, subject_idx=r.subject_idx, ethnicity=r.ethnicity,
                             attack=r.attack, label=int(r.attack == "real"),
                             split_2_1=r.split_2_1, split_2_2=r.split_2_2))
        if n % 500 == 0:
            print(f"  {n}/{len(video_df)} videos")
    FM_ = pd.DataFrame(rows); FM_.insert(0, "frame_id", np.arange(len(FM_)))
    arr = np.stack(imgs)
    np.save(npy, arr); FM_.to_csv(csv, index=False)
    return arr, FM_

def make_synthetic(n_per_eth=20, size=None, frames=3, seed=0):
    """Tiny fake dataset with the real schema. Split fractions mirror CeFA (40% / 20% / 40% of subjects)."""
    size = size or IMG
    rng = np.random.default_rng(seed)
    train_end, val_end = int(0.4 * n_per_eth), int(0.6 * n_per_eth)
    rows = []
    def add(sub, eth, atk, s21, s22):
        rows.append(dict(video_id=f"v{len(rows)}", path="", subject_idx=sub, ethnicity=eth, attack=atk,
                         split_2_1=s21, split_2_2=s22))
    for eth in "ACE":
        for sub in range(1, n_per_eth + 1):
            part = "train" if sub <= train_end else ("val" if sub <= val_end else "test")
            tv = part if part in ("train", "val") else ""
            add(sub, eth, "real", part, part)
            for _ in range(2):                                     # two print videos per subject
                add(sub, eth, "print", tv, "test" if part == "test" else "")
            add(sub, eth, "replay", "test" if part == "test" else "", tv)   # one replay video
    for i in range(40):                                            # 3D masks: test only
        add(i + 1, "3D", "3d", "test", "test")
    H = size; yy, xx = np.mgrid[0:H, 0:H] / H
    def pattern(atk):
        if atk == "real":   return 0.5 + 0.3 * np.sin(2 * np.pi * (xx + yy))
        if atk == "print":  return 0.5 + 0.3 * np.sign(np.sin(2 * np.pi * 4 * xx))
        if atk == "replay": return 0.5 + 0.3 * np.sign(np.sin(2 * np.pi * 4 * xx)) * np.sign(np.sin(2 * np.pi * 4 * yy))
        return 0.5 + 0.3 * np.exp(-((xx - 0.5) ** 2 + (yy - 0.5) ** 2) * 20)
    imgs, meta = [], []
    for r in rows:
        for _ in range(frames):
            im = pattern(r["attack"])[..., None] * (0.8 + 0.4 * rng.random(3)) + rng.normal(0, 0.08, (H, H, 3))
            imgs.append((np.clip(im, 0, 1) * 255).astype(np.uint8))
            meta.append(dict(r, label=int(r["attack"] == "real")))
    FM_ = pd.DataFrame(meta); FM_.insert(0, "frame_id", np.arange(len(FM_)))
    return np.stack(imgs), FM_

def frames_from_npz(video_df, size=None, k=None):
    """Read the per-protocol caches written by paper/code/experiments/cache_frames.py (--frames 5 --size 112)
    and return (frames, frame table) in manifest order. CeFA frames are already face crops on a black
    background; the cache pads each crop to a square and resizes it, so no face detector is involved."""
    size = size or IMG; k = k or FRAMES_PER_VIDEO
    by_path = {}
    for sp in SUBPROTOCOLS:
        f = CEFA_DIR / f"cache_p{sp.replace('_', '.')}_rgb{size}_f{k}.npz"
        z = np.load(f)
        for key in {n.split("/")[0] for n in z.files}:
            for pth, x in zip(z[f"{key}/path"], z[f"{key}/x"]):
                by_path.setdefault(str(pth), x)
    missing = [p for p in video_df.path if p not in by_path]
    assert not missing, f"{len(missing)} manifest videos are not in the caches, e.g. {missing[:3]}"
    imgs, rows = [], []
    for r in video_df.itertuples(index=False):
        x = by_path[r.path]
        assert x.shape == (k, size, size, 3)
        for im in x:
            imgs.append(im)
            rows.append(dict(video_id=r.video_id, subject_idx=r.subject_idx, ethnicity=r.ethnicity,
                             attack=r.attack, label=int(r.attack == "real"),
                             split_2_1=r.split_2_1, split_2_2=r.split_2_2))
    FM_ = pd.DataFrame(rows); FM_.insert(0, "frame_id", np.arange(len(FM_)))
    return np.stack(imgs), FM_

In [ ]:
if SYNTHETIC:
    FRAMES, FM = make_synthetic()
else:
    video_df = build_manifest()
    assert check_counts(video_df), "counts differ from the report's Table I: fix the manifest before training"
    FRAMES, FM = frames_from_npz(video_df)

assert (FM.frame_id.values == np.arange(len(FM))).all() and len(FM) == len(FRAMES)
print(FRAMES.shape, FRAMES.dtype, "|", len(FM), "frames |", FM.video_id.nunique(), "videos")
print(FM.drop_duplicates("video_id").attack.value_counts().to_dict())

## Step 4. Labeled sets, unlabeled pools and the splits

For each (sub-protocol, budget, seed, variant) `make_splits()` returns frame ids for:

* **labeled** frames: all frames of `budget` videos, sampled **at the video level** and stratified by class and ethnicity (so near-duplicate frames of one video can never sit in both the labeled and unlabeled sets);
* **unlabeled** frames: all other training videos, with labels never used;
* **U-visible** only: the training subjects' videos of the *other* attack instrument are added to the unlabeled pool (600 replay videos in 2_1, 1,200 print videos in 2_2 for real CeFA);
* **val** and **test** frames from the official validation and test subjects.

`budget="all"` labels every training video (the reference row in the report).

`audit_splits()` then asserts the properties that matter: subject-disjoint splits, no video in both labeled and unlabeled sets, labeled videos inside the training set, different seeds giving different subsets, and the expected pool sizes. **If it fails, nothing downstream is trustworthy.**

In [ ]:
def alloc_counts(sizes, n):
    """Largest-remainder allocation of n draws over strata, at least 1 per stratum, never more than the stratum size."""
    sizes = np.asarray(sizes, dtype=float)
    assert len(sizes) <= n <= sizes.sum(), f"cannot draw {n} videos from strata {sizes.tolist()}"
    quota = sizes / sizes.sum() * n
    base = np.minimum(np.maximum(1, np.floor(quota)), sizes).astype(int)
    while base.sum() > n:
        i = int(np.argmax(np.where(base > 1, base - quota, -np.inf))); base[i] -= 1
    while base.sum() < n:
        i = int(np.argmax(np.where(sizes - base > 0, quota - base, -np.inf))); base[i] += 1
    return base

def sample_labeled_videos(train_vids, n, seed):
    """Pick n training videos, stratified by (class, ethnicity). Deterministic given the seed."""
    rng = np.random.default_rng(seed)
    groups = [g for _, g in train_vids.groupby(["label", "ethnicity"], sort=True)]
    counts = alloc_counts([len(g) for g in groups], n)
    chosen = []
    for g, c in zip(groups, counts):
        chosen += list(rng.choice(g.video_id.values, size=c, replace=False))
    return chosen

def make_splits(fm, sp, budget, seed, variant="hidden"):
    col = f"split_{sp}"
    vids = fm.drop_duplicates("video_id")
    train_v = vids[vids[col] == "train"]
    train_attacks = set(train_v[train_v.label == 0].attack)
    assert len(train_attacks) == 1, f"expected exactly one attack instrument in training, got {train_attacks}"
    train_frames = fm[fm[col] == "train"]
    if budget == "all":
        lab_ids = set(train_v.video_id)
        labeled, unlabeled = train_frames, train_frames          # the paper also uses all images as unlabeled
    else:
        lab_ids = set(sample_labeled_videos(train_v, int(budget), seed))
        labeled = train_frames[train_frames.video_id.isin(lab_ids)]
        unlabeled = train_frames[~train_frames.video_id.isin(lab_ids)]
    if variant == "visible":
        other = "replay" if "print" in train_attacks else "print"
        train_subjects = set(zip(train_v.ethnicity, train_v.subject_idx))
        is_train_subject = pd.Series(list(zip(fm.ethnicity, fm.subject_idx)), index=fm.index).isin(train_subjects)
        extra = fm[(fm.attack == other) & is_train_subject]
        unlabeled = pd.concat([unlabeled, extra])
    return dict(lab_idx=labeled.frame_id.values.astype(np.int64),
                unl_idx=unlabeled.frame_id.values.astype(np.int64),
                val_idx=fm[fm[col] == "val"].frame_id.values.astype(np.int64),
                test_idx=fm[fm[col] == "test"].frame_id.values.astype(np.int64),
                lab_videos=sorted(lab_ids))

In [ ]:
def audit_splits(fm=None, budgets=None, seeds=None):
    fm = FM if fm is None else fm
    budgets = (BUDGETS if budgets is None else budgets) + ["all"]
    seeds = SEEDS if seeds is None else seeds
    rows = []
    for sp in SUBPROTOCOLS:
        col = f"split_{sp}"
        subj = lambda s: set(zip(fm[fm[col] == s].ethnicity, fm[fm[col] == s].subject_idx))
        tr, va, te = subj("train"), subj("val"), subj("test")
        assert not (tr & va) and not (tr & te) and not (va & te), f"{sp}: subjects overlap between splits"
        vid = fm.drop_duplicates("video_id")
        n_train_videos = int((vid[col] == "train").sum())
        for b in budgets:
            for v in ["hidden", "visible"]:
                subsets = []
                for s in seeds:
                    d = make_splits(fm, sp, b, s, v)
                    lab_v = set(fm.loc[d["lab_idx"], "video_id"]); unl_v = set(fm.loc[d["unl_idx"], "video_id"])
                    assert set(d["lab_idx"]) <= set(fm[fm[col] == "train"].frame_id), "labeled frame outside training set"
                    if b != "all":
                        assert not (lab_v & unl_v), "a video is in both the labeled and the unlabeled pool"
                        assert len(lab_v) == b, f"asked for {b} labeled videos, got {len(lab_v)}"
                        labeled_classes = fm.loc[d["lab_idx"], "label"].unique()
                        assert len(labeled_classes) == 2, "labeled set misses a class"
                    assert not (set(d["unl_idx"]) & (set(d["val_idx"]) | set(d["test_idx"]))), "unlabeled data overlaps val/test"
                    assert len(d["val_idx"]) > 0 and len(d["test_idx"]) > 0
                    subsets.append(tuple(d["lab_videos"]))
                    if s == seeds[0]:
                        rows.append(dict(sp=sp, budget=b, variant=v, lab_videos=len(lab_v) if b != "all" else n_train_videos,
                                         lab_frames=len(d["lab_idx"]), unl_videos=len(unl_v), unl_frames=len(d["unl_idx"]),
                                         val_frames=len(d["val_idx"]), test_frames=len(d["test_idx"])))
                if b != "all" and len(seeds) > 1:
                    assert len(set(subsets)) == len(subsets), "different seeds gave identical labeled sets"
    out = pd.DataFrame(rows)
    print(out.to_string(index=False))
    print("\nAll split checks passed.")
    return out

_ = audit_splits()

## Step 5. PyTorch setup, augmentation and models

Augmentation runs on the GPU in one batched call: random translation (and flip when enabled) through an affine grid, then Gaussian noise, as in the paper's Table 6. The teacher and the student each get their **own independent augmentation** of the same images (the noise terms $\eta$ and $\eta'$ in Eq. 1).

The next cell also prints the parameter counts of the two CeFA models. The report currently says about 1.5M and 23.5M; **if the printed numbers differ, change the report.**

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
dev = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
def sync():
    if dev.type == "cuda": torch.cuda.synchronize()
    elif dev.type == "mps": torch.mps.synchronize()
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device", dev)

def seed_all(s):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def prep_eval(x_uint8, mean=None, std=None):
    """uint8 (B,H,W,3) -> normalized float (B,3,H,W)."""
    m = torch.tensor(IMAGENET_MEAN if mean is None else mean, device=x_uint8.device).view(1, 3, 1, 1)
    s = torch.tensor(IMAGENET_STD if std is None else std, device=x_uint8.device).view(1, 3, 1, 1)
    return (x_uint8.permute(0, 3, 1, 2).float() / 255.0 - m) / s

def augment(x_uint8, cfg, mean=None, std=None):
    """Random translation (+ optional flip) via an affine grid, then Gaussian noise. Independent per sample."""
    x = prep_eval(x_uint8, mean, std)
    B, _, H, W = x.shape
    t = cfg["translate_px"] * 2.0 / W                       # pixels -> normalized [-1, 1] units
    tx = (torch.rand(B, device=x.device) * 2 - 1) * t
    ty = (torch.rand(B, device=x.device) * 2 - 1) * t
    sx = 1.0 - 2.0 * (torch.rand(B, device=x.device) < 0.5).float() if cfg["flip"] else torch.ones(B, device=x.device)
    theta = torch.zeros(B, 2, 3, device=x.device)
    theta[:, 0, 0] = sx; theta[:, 1, 1] = 1.0; theta[:, 0, 2] = tx; theta[:, 1, 2] = ty
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, padding_mode="reflection", align_corners=False)
    return x + cfg["noise_sigma"] * torch.randn_like(x)

Xt = torch.from_numpy(FRAMES)                                # shares memory with FRAMES
Y_ALL = torch.as_tensor(FM.label.values).long()              # 1 = live, 0 = attack

In [ ]:
def make_student(pretrained=True):
    w = "IMAGENET1K_V1" if (pretrained and not SYNTHETIC) else None
    m = torchvision.models.mobilenet_v3_small(weights=w)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
    return m

def make_teacher(pretrained=True):
    w = "IMAGENET1K_V2" if (pretrained and not SYNTHETIC) else None
    m = torchvision.models.resnet50(weights=w)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m

def n_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"MobileNetV3-Small: {n_params(make_student(False)) / 1e6:.2f}M params | "
      f"ResNet-50: {n_params(make_teacher(False)) / 1e6:.2f}M params   (report estimates: 1.5M and 23.5M)")

## Step 6. Mean Teacher utilities

Four small functions, each tied to an equation in the report:

* `rampup` is Eq. 4: $\exp[-5(1-\min(t/T_r,1))^2]$.
* `ema_update` is Eq. 2: $\theta'_t=\alpha\theta'_{t-1}+(1-\alpha)\theta_t$, with a decay warm-up $\min(1-1/(t+1),\alpha)$ (so the teacher equals the student at step 0). **BatchNorm statistics are copied from the student**; this is the choice the report says to confirm.
* `consistency` is Eq. 1 (MSE between the two softmax outputs, averaged over samples and classes) or the temperature-scaled KL variant (Eq. 5).
* `freeze` turns a model into a fixed teacher (E4).

**Loss scale.** Because the MSE is averaged over samples and classes, sensible values of `lam_c` are much larger than 1 (public Mean Teacher code uses values on the order of 10 to 100 with this normalisation, which I am recalling from memory, so do not trust it). Sweep `{1, 10, 100}` on the validation subjects in Step 10 and use the winner.

In [ ]:
def rampup(step, ramp_steps):
    x = min(step / max(ramp_steps, 1), 1.0)
    return math.exp(-5.0 * (1.0 - x) ** 2)

@torch.no_grad()
def ema_update(teacher, student, alpha, step):
    a = min(1 - 1 / (step + 1), alpha)
    for pt, ps in zip(teacher.parameters(), student.parameters()):
        pt.mul_(a).add_(ps.detach(), alpha=1 - a)
    for bt, bs in zip(teacher.buffers(), student.buffers()):
        bt.copy_(bs)                      # BatchNorm statistics: copied from the student

def consistency(logits_s, logits_t, kind="mse", tau=2.0):
    if kind == "mse":
        return F.mse_loss(F.softmax(logits_s, 1), F.softmax(logits_t, 1))
    log_ps = F.log_softmax(logits_s / tau, 1)
    pt = F.softmax(logits_t / tau, 1)
    return (tau ** 2) * F.kl_div(log_ps, pt, reduction="batchmean")   # = tau^2 * KL(p_T || p_S)

def freeze(m):
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m

## Step 7. Smoke tests: run these before any long run

Eight checks that finish in seconds to a minute. They verify the pieces I could not run myself: the ramp-up values, the EMA formula and warm-up, the consistency losses, that cross-entropy sees only labeled samples, that augmentation changes the input and keeps its shape, that freezing works, and that the student can overfit a tiny batch (which catches broken labels or a bad learning rate).

**If any assertion fails, stop and fix it.** Do not interpret results from a pipeline that fails these.

In [ ]:
def smoke_tests(overfit_steps=150):
    # 1. ramp-up values
    assert abs(rampup(0, 100) - math.exp(-5)) < 1e-9
    assert abs(rampup(50, 100) - math.exp(-5 * 0.25)) < 1e-9
    assert rampup(100, 100) == 1.0 and rampup(10_000, 100) == 1.0
    # 2. EMA equals the formula; 3. warm-up makes the teacher equal the student at step 0
    s = nn.Linear(3, 2); t = copy.deepcopy(s)
    with torch.no_grad():
        for p in s.parameters(): p.add_(1.0)
    before = [p.clone() for p in t.parameters()]
    ema_update(t, s, alpha=0.9, step=10_000)
    for pb, pt, ps in zip(before, t.parameters(), s.parameters()):
        assert torch.allclose(pt, 0.9 * pb + 0.1 * ps, atol=1e-6)
    ema_update(t, s, alpha=0.999, step=0)
    for pt, ps in zip(t.parameters(), s.parameters()):
        assert torch.allclose(pt, ps)
    # 4. consistency losses
    z = torch.randn(8, 2)
    assert consistency(z, z, "mse").item() < 1e-12 and abs(consistency(z, z, "kl").item()) < 1e-6
    assert consistency(z, z + torch.randn(8, 2), "mse").item() > 0 and consistency(z, z + torch.randn(8, 2), "kl").item() > 0
    # 5. cross-entropy must see labeled samples only
    logits = torch.randn(10, 2, requires_grad=True)
    F.cross_entropy(logits[:4], torch.tensor([0, 1, 0, 1])).backward()
    assert logits.grad[4:].abs().sum().item() == 0 and logits.grad[:4].abs().sum().item() > 0
    # 6. augmentation: shape, range sanity, and randomness
    x = Xt[:8].to(dev)
    a1, a2 = augment(x, CFG), augment(x, CFG)
    assert a1.shape == (8, 3, x.shape[1], x.shape[2]) and not torch.allclose(a1, a2)
    assert torch.allclose(prep_eval(x), prep_eval(x))
    # 7. freezing
    m = freeze(make_student(False)); assert not m.training and all(not p.requires_grad for p in m.parameters())
    # 8. the student can overfit 64 frames
    seed_all(0)
    idx = np.random.default_rng(0).choice(len(FM), 64, replace=False)
    xb, yb = Xt[torch.as_tensor(idx)].to(dev), Y_ALL[torch.as_tensor(idx)].to(dev)
    assert yb.unique().numel() == 2, "the 64 sampled frames contain one class only; rerun"
    net = make_student(False).to(dev).train()
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3)
    for _ in range(overfit_steps):
        loss = F.cross_entropy(net(prep_eval(xb)), yb); opt.zero_grad(); loss.backward(); opt.step()
    # torchvision's MobileNetV3 uses BatchNorm momentum 0.01, so after a few steps its running statistics are still
    # stale. Check the fit with batch statistics (train mode) and dropout switched off instead of eval mode.
    net.train()
    for m_ in net.modules():
        if isinstance(m_, nn.Dropout): m_.eval()
    with torch.no_grad():
        acc = (net(prep_eval(xb)).argmax(1) == yb).float().mean().item()
    print(f"overfit accuracy on 64 frames: {acc:.2f}")
    assert acc > 0.9, "student cannot overfit 64 frames: check labels, lr, data"
    print("All smoke tests passed.")

smoke_tests()

## Step 8. E0: SVHN implementation check

This reproduces the paper's setting as closely as the budget allows: the 13-layer ConvNet (Table 6), 1,000 labels (class-balanced), **all training images as the unlabeled set**, Adam with learning rate 0.003, MSE consistency, EMA decay 0.99 then 0.999, ramp-up over 22% of training. Convolutions use weight normalisation and a mean-only batch normalisation, as the paper describes.

What is **not** the paper's: the schedule is shortened (the paper trains 180,000 steps), the learning rate is constant, and the labeled/unlabeled split of each batch (`batch_lab`, `batch_unl`) is my default. **Check the paper's appendix and the authors' SVHN config and set these to match, and document any difference in the report.**

Compare your numbers with the paper's Table 1 values (1,000 labels): supervised-only 12.32 ± 0.95, Pi model 4.95 ± 0.26, Mean Teacher 3.95 ± 0.19. With a short schedule you will probably be worse than the paper; what matters is the *ordering* and a plausible margin, and that you say so honestly. Run `sup` and `mt` first, then `pi` if time allows. Set `SV["steps"]` from a quick timing run.

In [ ]:
class MeanOnlyBN(nn.Module):
    """Batch normalization that subtracts the batch mean only (no variance scaling), plus a learned bias."""
    def __init__(self, c, momentum=0.1):
        super().__init__()
        self.register_buffer("running_mean", torch.zeros(c))
        self.bias = nn.Parameter(torch.zeros(c)); self.momentum = momentum
    def forward(self, x):
        if self.training:
            m = x.mean(dim=(0, 2, 3))
            with torch.no_grad():
                self.running_mean.mul_(1 - self.momentum).add_(m.detach() * self.momentum)
        else:
            m = self.running_mean
        return x - m[None, :, None, None] + self.bias[None, :, None, None]

def _conv(cin, cout, k, pad):
    conv = nn.utils.parametrizations.weight_norm(nn.Conv2d(cin, cout, k, padding=pad))
    return nn.Sequential(conv, MeanOnlyBN(cout), nn.LeakyReLU(0.1))

class ConvNet13(nn.Module):
    """The paper's 13-layer ConvNet (Table 6)."""
    def __init__(self, n_classes=10):
        super().__init__()
        L = [_conv(3, 128, 3, 1), _conv(128, 128, 3, 1), _conv(128, 128, 3, 1), nn.MaxPool2d(2), nn.Dropout(0.5),
             _conv(128, 256, 3, 1), _conv(256, 256, 3, 1), _conv(256, 256, 3, 1), nn.MaxPool2d(2), nn.Dropout(0.5),
             _conv(256, 512, 3, 0), _conv(512, 256, 1, 0), _conv(256, 128, 1, 0),
             nn.AdaptiveAvgPool2d(1), nn.Flatten(),
             nn.utils.parametrizations.weight_norm(nn.Linear(128, n_classes))]
        self.net = nn.Sequential(*L)
    def forward(self, x):
        return self.net(x)

_t = ConvNet13()
assert _t(torch.randn(2, 3, 32, 32)).shape == (2, 10)
print(f"ConvNet13: {n_params(_t) / 1e6:.2f}M params (report estimate: about 3.1M)")
del _t

def load_svhn():
    if SYNTHETIC:
        rng = np.random.default_rng(0)
        def fake(n):
            y = rng.integers(0, 10, n)
            off = np.stack([(y * 25) % 255, (y * 60) % 255, (y * 95) % 255], 1)[:, None, None, :]
            return np.clip(off + rng.normal(0, 30, (n, 32, 32, 3)), 0, 255).astype(np.uint8), y
        return (*fake(2000), *fake(500))
    from torchvision.datasets import SVHN
    tr = SVHN(str(ROOT / "svhn"), split="train", download=True)
    te = SVHN(str(ROOT / "svhn"), split="test", download=True)
    return (np.transpose(tr.data, (0, 2, 3, 1)).copy(), np.asarray(tr.labels),
            np.transpose(te.data, (0, 2, 3, 1)).copy(), np.asarray(te.labels))

SVHN_DATA = load_svhn()
print("SVHN:", SVHN_DATA[0].shape, SVHN_DATA[2].shape)

In [ ]:
SV = dict(n_labels=1000, steps=3000, batch_lab=50, batch_unl=50,     # batch split: check the paper's appendix
          lr=0.003, lam_c=10.0, ramp_frac=40000 / 180000, alpha_early=0.99, alpha_late=0.999,
          translate_px=2, flip=False, noise_sigma=0.15, cons_loss="mse", tau=2.0)
if SYNTHETIC:
    SV.update(n_labels=100, steps=150)

@torch.no_grad()
def svhn_error(model, X, y, mean, std, bs=500):
    model.eval(); wrong = 0
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i + bs]).to(dev)
        wrong += (model(prep_eval(xb, mean, std)).argmax(1).cpu().numpy() != y[i:i + bs]).sum()
    return 100.0 * wrong / len(X)

def run_svhn(mode, seed, cfg=SV):
    name = f"svhn_{mode}_{cfg['n_labels']}_s{seed}"; out = RESULTS / f"{name}.json"
    if out.exists():
        print("skip (done):", name); return json.load(open(out))
    seed_all(seed)
    Xtr, ytr, Xte, yte = SVHN_DATA
    mean = (Xtr.reshape(-1, 3).mean(0) / 255).tolist(); std = (Xtr.reshape(-1, 3).std(0) / 255).tolist()
    rng = np.random.default_rng(seed)
    lab = np.concatenate([rng.choice(np.where(ytr == c)[0], cfg["n_labels"] // 10, replace=False) for c in range(10)])
    unl = np.arange(len(Xtr))                                   # the paper uses all training images as unlabeled
    Xtt, ytt = torch.from_numpy(Xtr), torch.as_tensor(ytr).long()
    student = ConvNet13().to(dev)
    opt = torch.optim.Adam(student.parameters(), lr=cfg["lr"], betas=(0.9, 0.999))
    teacher = freeze(copy.deepcopy(student)) if mode == "mt" else None
    ramp_steps = max(1, int(cfg["ramp_frac"] * cfg["steps"]))
    t0 = time.time()
    for step in range(cfg["steps"]):
        student.train()
        if mode == "sup":
            idx = rng.choice(lab, cfg["batch_lab"] + cfg["batch_unl"], replace=True); n_l = len(idx)
        else:
            il = rng.choice(lab, cfg["batch_lab"], replace=True); iu = rng.choice(unl, cfg["batch_unl"], replace=False)
            idx = np.concatenate([il, iu]); n_l = len(il)
        ti = torch.as_tensor(idx); xb, yb = Xtt[ti].to(dev), ytt[ti].to(dev)
        logits_s = student(augment(xb, cfg, mean, std))
        loss = F.cross_entropy(logits_s[:n_l], yb[:n_l])
        if mode != "sup":
            xt = augment(xb, cfg, mean, std)
            if mode == "pi":
                logits_t = student(xt)
            else:
                with torch.no_grad(): logits_t = teacher(xt)
            loss = loss + cfg["lam_c"] * rampup(step, ramp_steps) * consistency(logits_s, logits_t, cfg["cons_loss"], cfg["tau"])
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if mode == "mt":
            ema_update(teacher, student, cfg["alpha_early"] if step < ramp_steps else cfg["alpha_late"], step)
    ev = teacher if mode == "mt" else student
    rec = dict(exp="E0", mode=mode, dataset="svhn", budget=cfg["n_labels"], seed=seed, steps=cfg["steps"],
               error_pct=float(svhn_error(ev, Xte, yte, mean, std)), seconds=round(time.time() - t0, 1))
    json.dump(rec, open(out, "w"), default=float)
    print(name, "test error %.2f%%" % rec["error_pct"], "in", rec["seconds"], "s")
    return rec

PAPER_SVHN = {"sup": (12.32, 0.95), "pi": (4.95, 0.26), "mt": (3.95, 0.19)}   # paper's Table 1, 1,000 labels

def svhn_table():
    rows = []
    for m, (pm, ps) in PAPER_SVHN.items():
        rs = [json.load(open(p))["error_pct"] for p in sorted(RESULTS.glob(f"svhn_{m}_*.json"))]
        mine = "-" if not rs else (f"{np.mean(rs):.2f}" if len(rs) == 1 else f"{np.mean(rs):.2f} +- {np.std(rs, ddof=1):.2f}")
        rows.append(dict(method=m, paper=f"{pm:.2f} +- {ps:.2f}", mine=mine, runs=len(rs)))
    return pd.DataFrame(rows)

# Run order: supervised-only and Mean Teacher first; add "pi" if time allows.
for s in SEEDS:
    for m in ["sup", "mt", "pi"]:
        run_svhn(m, s)
svhn_table()

## Step 9. The CeFA training function

`run_experiment(mode, sp, budget, seed, variant)` runs one experiment and writes one JSON file.

| `mode` | experiment | teacher | consistency | evaluated model |
|---|---|---|---|---|
| `sup` | E1 | none | no | student |
| `pi` | E2 | the student itself, gradients through both passes | yes | student |
| `mt` | E3 | EMA of the student | yes | **EMA teacher** (as in the paper) |
| `kd` | E4 | frozen ResNet-50 fine-tuned on the labeled subset only | yes | **student** (the ResNet is only a teacher) |

Things worth knowing before you read the code:

* A step uses `batch_lab` labeled and `batch_unl` unlabeled frames. Cross-entropy uses the labeled ones only; the consistency loss uses all of them. Labeled frames are drawn with replacement because the labeled set is small.
* While training, the notebook logs **validation** error every `eval_every` steps. **The test subjects are used exactly once, at the end.** Do not add test curves: they would let you pick settings on the test set.
* The decision threshold is the equal-error point on the validation videos. Video scores are the mean of the frame scores.
* For `mt`, the student's ACER is also stored as `acer_student`, so you can see whether the EMA teacher actually helps.

In [ ]:
EXP = {"sup": "E1", "pi": "E2", "mt": "E3", "kd": "E4"}

@torch.no_grad()
def predict(model, frame_ids, bs=256):
    model.eval(); out = []
    for i in range(0, len(frame_ids), bs):
        xb = Xt[torch.as_tensor(frame_ids[i:i + bs])].to(dev)
        out.append(F.softmax(model(prep_eval(xb)), dim=1)[:, 1].float().cpu())
    return torch.cat(out).numpy()

def video_level(frame_rows, scores):
    d = frame_rows[["video_id", "label", "attack"]].copy(); d["score"] = scores
    return d.groupby("video_id").agg(score=("score", "mean"), label=("label", "first"), attack=("attack", "first")).reset_index()

def score_split(model, frame_ids):
    return video_level(FM.loc[frame_ids], predict(model, frame_ids))

def val_error(model, val_idx):
    v = score_split(model, val_idx)
    return float(np.mean((v.score.values >= 0.5).astype(int) != v.label.values))

@torch.no_grad()
def recalibrate_bn(model, lab_idx, seed, n_batches=100, bs=64):
    """Copy of `model` with BatchNorm running statistics recomputed on CLEAN labeled training frames.

    Training adds Gaussian noise (noise_sigma) to every input, so the running statistics describe
    noisy frames, while evaluation uses clean frames. Without this step the supervised baseline
    collapsed in several runs (diagnostic of 15 Nov: ACER 48.11% -> 11.78% after recalibration).
    Weights are not changed and no validation or test frame is used.
    """
    m = copy.deepcopy(model)
    bns = [x for x in m.modules() if isinstance(x, nn.modules.batchnorm._BatchNorm)]
    if not bns:
        return m
    for x in bns:
        x.reset_running_stats(); x.momentum = None        # cumulative average over the batches below
    m.eval()
    for x in bns:
        x.train()          # BatchNorm only: dropout stays off, so no random numbers are drawn here
    rng = np.random.default_rng(seed + 3000)
    for _ in range(n_batches):
        ti = torch.as_tensor(rng.choice(lab_idx, bs, replace=True))
        m(prep_eval(Xt[ti].to(dev)))
    return m.eval()

def fas_metrics(val_v, test_v):
    groups = np.where(test_v.label.values == 1, "real", test_v.attack.values)
    return evaluate(val_v.score.values, val_v.label.values, test_v.score.values, test_v.label.values, groups)

def train_frozen_teacher(lab_idx, cfg, seed):
    """Fine-tune ResNet-50 on the labeled frames only, then freeze it (E4)."""
    t = make_teacher().to(dev)
    opt = torch.optim.AdamW(t.parameters(), lr=cfg["teacher_lr"], weight_decay=cfg["weight_decay"])
    rng = np.random.default_rng(seed + 2000)
    for step in range(cfg["teacher_steps"]):
        t.train()
        ti = torch.as_tensor(rng.choice(lab_idx, cfg["batch_lab"] * 2, replace=True))
        loss = F.cross_entropy(t(augment(Xt[ti].to(dev), cfg)), Y_ALL[ti].to(dev))
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return freeze(t)

def run_experiment(mode, sp, budget, seed, variant="hidden", cfg=None, tag="", timing_only=False):
    cfg = CFG if cfg is None else cfg
    name = f"{mode}_{sp}_{budget}_{variant}_s{seed}" + (f"_{tag}" if tag else "")
    out = RESULTS / f"{name}.json"
    if out.exists() and not timing_only:
        print("skip (done):", name); return json.load(open(out))
    seed_all(seed)
    d = make_splits(FM, sp, budget, seed, variant)
    lab, unl = d["lab_idx"], d["unl_idx"]
    student = make_student().to(dev)
    opt = torch.optim.AdamW(student.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["steps"])
    teacher = None
    if mode == "mt":
        teacher = freeze(copy.deepcopy(student))
    elif mode == "kd":
        teacher = train_frozen_teacher(lab, cfg, seed)
    ramp_steps = max(1, int(cfg["ramp_frac"] * cfg["steps"]))
    rng = np.random.default_rng(seed + 1000)
    curve = []
    sync()
    t0 = time.time()
    for step in range(cfg["steps"]):
        student.train()
        il = rng.choice(lab, cfg["batch_lab"], replace=True)
        if mode == "sup":
            idx = il
        else:
            iu = rng.choice(unl, cfg["batch_unl"], replace=len(unl) < cfg["batch_unl"])
            idx = np.concatenate([il, iu])
        n_l = len(il); ti = torch.as_tensor(idx)
        xb, yb = Xt[ti].to(dev), Y_ALL[ti].to(dev)
        logits_s = student(augment(xb, cfg))
        loss_ce = F.cross_entropy(logits_s[:n_l], yb[:n_l])
        loss_c = torch.zeros((), device=dev)
        if mode != "sup":
            xt = augment(xb, cfg)
            if mode == "pi":
                logits_t = student(xt)                      # gradients flow through both passes
            else:
                with torch.no_grad(): logits_t = teacher(xt)
            loss_c = consistency(logits_s, logits_t, cfg["cons_loss"], cfg["tau"])
        loss = loss_ce + cfg["lam_c"] * rampup(step, ramp_steps) * loss_c
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if mode == "mt":
            ema_update(teacher, student, cfg["alpha_early"] if step < ramp_steps else cfg["alpha_late"], step)
        if not timing_only and ((step + 1) % cfg["eval_every"] == 0 or step + 1 == cfg["steps"]):
            ev = recalibrate_bn(teacher if mode == "mt" else student, lab, seed, n_batches=25)
            curve.append([step + 1, val_error(ev, d["val_idx"]), float(loss_ce.item()), float(loss_c.item())])
    sync()
    seconds = time.time() - t0
    if timing_only:
        return seconds / cfg["steps"]
    ev_raw = teacher if mode == "mt" else student
    raw = fas_metrics(score_split(ev_raw, d["val_idx"]), score_split(ev_raw, d["test_idx"]))
    ev = recalibrate_bn(ev_raw, lab, seed)
    met = fas_metrics(score_split(ev, d["val_idx"]), score_split(ev, d["test_idx"]))
    rec = dict(exp=EXP[mode], mode=mode, dataset="cefa", sub_protocol=sp, budget=budget, seed=seed, variant=variant,
               tag=tag, n_lab_frames=int(len(lab)), n_unl_frames=int(len(unl)), seconds=round(seconds, 1),
               lam_c=cfg["lam_c"], steps=cfg["steps"], bn_recalibrated=True, curve=curve, **met,
               acer_raw=raw["acer"], val_eer_raw=raw["val_eer"])
    if mode == "mt":
        st = recalibrate_bn(student, lab, seed)
        rec["acer_student"] = fas_metrics(score_split(st, d["val_idx"]), score_split(st, d["test_idx"]))["acer"]
    json.dump(rec, open(out, "w"), default=float)
    print(f"{name}: ACER {100 * rec['acer']:.2f}%  (val EER {100 * rec['val_eer']:.2f}%; before BN recal. {100 * rec['acer_raw']:.2f}%)  {rec['seconds']}s")
    del teacher
    if dev.type == "cuda": torch.cuda.empty_cache()
    elif dev.type == "mps": torch.mps.empty_cache()
    return rec

## Step 10. Timing, then choosing `steps` and `lam_c`

**Timing.** Run this first and read the seconds per step. Then set `CFG["steps"]` so one run takes the time you can afford (for example 30-45 minutes), and multiply by the number of runs in your grid to see whether it fits. The frozen-teacher fine-tuning (`teacher_steps`) is extra time for E4.

**Tuning.** `sweep_lam_c` runs Mean Teacher for several consistency weights on **one seed and one budget** and reports the **validation** EER (not the test metrics). Pick the best, set `CFG["lam_c"]`, and say in the report how you chose it. If the differences are smaller than the seed-to-seed spread, say that too.

In [ ]:
def time_run(mode, sp="2_1", budget=None, n=30):
    budget = budget if budget is not None else BUDGETS[-1]
    cfg = dict(CFG, steps=n, eval_every=10 ** 9)
    s = run_experiment(mode, sp, budget, 0, cfg=cfg, tag="timing", timing_only=True)
    print(f"{mode:3s}: {1000 * s:.0f} ms/step  ->  {CFG['steps']} steps = {s * CFG['steps'] / 60:.1f} min per run")
    return s

for m in ["sup", "mt"]:
    time_run(m)

def sweep_lam_c(values=(1, 10, 100), sp="2_1", budget=None, seed=0):
    budget = budget if budget is not None else BUDGETS[-1]
    rows = []
    for lam in values:
        r = run_experiment("mt", sp, budget, seed, cfg=dict(CFG, lam_c=lam), tag=f"lam{lam}")
        rows.append(dict(lam_c=lam, val_eer_pct=round(100 * r["val_eer"], 2)))
    out = pd.DataFrame(rows); print(out.to_string(index=False)); return out

sweep_lam_c()        # run once, then set CFG["lam_c"] to the value with the lowest validation EER

## Step 11. Run E1, E3 and E4

Priority order if time is short: **E1 and E3 first** (the replication), then E4, then E2. Each call is skipped if its JSON already exists, so you can rerun this cell after a disconnect. The grid is sub-protocol x budget x seed, plus the `all` reference budget when you want that column of the report's Table IV.

Do not change `CFG` between runs that you intend to compare. If you must, record the change.

In [ ]:
def run_grid(modes, sps=None, budgets=None, seeds=None, variant="hidden"):
    sps = SUBPROTOCOLS if sps is None else sps
    budgets = BUDGETS if budgets is None else budgets
    seeds = SEEDS if seeds is None else seeds
    for sp in sps:
        for b in budgets:
            for s in seeds:
                for m in modes:
                    run_experiment(m, sp, b, s, variant)

run_grid(["sup", "mt"])        # E1, E3
run_grid(["kd"])               # E4: frozen ResNet-50 teacher
# run_grid(["pi"])             # E2: Pi model (optional)
run_grid(["sup", "mt", "kd"], budgets=["all"], seeds=[0])   # reference column, one seed only (time)

## Step 12. E5: U-visible vs U-hidden

`variant="hidden"` (the runs above): the unlabeled pool is the remaining training videos of the *same* attack instrument as the labeled ones.
`variant="visible"`: the unlabeled pool also contains the training subjects' videos of the *other* instrument (labels never used). The test set is the same in both.

Only methods that use unlabeled data can differ (E3 and E4); E1 ignores the pool. The `hidden` runs already exist from Step 11 and are skipped. Compare `apcer_replay` (for 2_1) or `apcer_print` (for 2_2), and `apcer_3d`, between the two variants. **This protocol is my own modification of CeFA's official one; say so wherever you report it.**

In [ ]:
for v in ["hidden", "visible"]:
    run_grid(["mt"], budgets=[BUDGETS[-1]], variant=v)         # E3
    # run_grid(["kd"], budgets=[BUDGETS[-1]], variant=v)       # E4

## Step 13. Tables and figures for the report

* `table_cefa` builds the report's **Table IV** (ACER in %, mean $\pm$ std over seeds; with one seed only the mean is shown, and you must say so in the caption). The "mean over sub-protocols" columns average 2_1 and 2_2 within each seed first, as CeFA does.
* `latex_rows` prints the rows ready to paste into the LaTeX table.
* `table_attacks` gives APCER per attack type (print, replay, 3D) for the discussion.
* `plot_curves` saves a PDF of the **validation**-error curves. Note that the report's placeholder caption says "test error"; change it to **validation error**.
* With only 3 seeds, compare methods by per-seed differences on the same label subsets, and do not claim significance.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

def load_results(tag=""):
    rows = [json.load(open(p)) for p in sorted(RESULTS.glob("*.json"))]
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["tag"] = df["tag"].fillna("") if "tag" in df else ""
    df["budget"] = df["budget"].astype(str)
    return df[df.tag == tag].copy()

def fmt(x, scale=100.0):
    x = np.asarray(list(x), dtype=float) * scale
    if len(x) == 0: return "-"
    return f"{x.mean():.2f}" if len(x) == 1 else f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

def table_cefa(df, metric="acer", variant="hidden", exps=("E1", "E3", "E4")):
    d = df[(df.dataset == "cefa") & (df.variant == variant)]
    cols = [(sp, str(b)) for sp in SUBPROTOCOLS for b in list(BUDGETS) + ["all"]]
    rows = []
    for e in exps:
        row = {"method": e}
        for sp, b in cols:
            row[f"{sp}/{b}"] = fmt(d[(d.exp == e) & (d.sub_protocol == sp) & (d.budget == b)][metric])
        for b in map(str, BUDGETS):
            sub = d[(d.exp == e) & (d.budget == b)]
            g = sub.groupby("seed")[metric].agg(["mean", "count"]) if len(sub) else pd.DataFrame(columns=["mean", "count"])
            row[f"mean/{b}"] = fmt(g[g["count"] == len(SUBPROTOCOLS)]["mean"])
        rows.append(row)
    return pd.DataFrame(rows).set_index("method")

def latex_rows(tbl):
    names = {"E1": "E1 Supervised-only", "E2": "E2 Pi model", "E3": "E3 Mean Teacher (EMA)", "E4": "E4 Frozen ResNet-50 teacher"}
    for e, r in tbl.iterrows():
        print(names.get(e, e) + " & " + " & ".join(v.replace("±", r"$\pm$") for v in r.values) + r" \\")

def table_attacks(df, variant="hidden"):
    d = df[(df.dataset == "cefa") & (df.variant == variant)]
    cols = [c for c in ["apcer_print", "apcer_replay", "apcer_3d", "apcer_silicone"] if c in d.columns]
    return d.groupby(["exp", "sub_protocol", "budget"])[cols].mean().mul(100).round(2)

def plot_curves(df, sp="2_1", budget=None, exps=("E1", "E3"), variant="hidden", save=True):
    budget = str(budget if budget is not None else BUDGETS[-1])
    fig, ax = plt.subplots(figsize=(4.2, 3.0))
    for e in exps:
        sub = df[(df.dataset == "cefa") & (df.exp == e) & (df.sub_protocol == sp) & (df.budget == budget) & (df.variant == variant)]
        if sub.empty:
            continue
        steps = np.array([c[0] for c in sub.iloc[0]["curve"]])
        errs = np.array([[c[1] for c in cv] for cv in sub["curve"]]) * 100
        m = errs.mean(0); s = errs.std(0, ddof=1) if len(errs) > 1 else np.zeros_like(m)
        ax.plot(steps, m, label=e); ax.fill_between(steps, m - s, m + s, alpha=0.2)
    ax.set_xlabel("training step"); ax.set_ylabel("validation error (%)"); ax.legend()
    ax.set_title(f"CeFA {sp}, {budget} labeled videos")
    fig.tight_layout()
    if save:
        fig.savefig(ROOT / f"curves_{sp}_{budget}.pdf")
    return fig

df = load_results()
if not df.empty:
    tbl = table_cefa(df)
    print(tbl.to_string()); print(); latex_rows(tbl); print()
    print(table_attacks(df).to_string())
    _ = plot_curves(df)
    plt.show()
else:
    print("no results yet")

## Step 14. Efficiency numbers for Table V

Parameters, multiply-accumulates (counted by PyTorch's FLOP counter, divided by two) and CPU latency at batch size 1 with one thread. **Compare the ConvNet MACs with my hand count in the report** (0.73 GMACs at 32x32 and 9.3 at 112x112); if the counter disagrees noticeably, trust the counter and fix the report. Latency depends on the machine: state which CPU you measured on.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode

def macs_and_latency(model, size, runs=30):
    model = model.eval().cpu(); x = torch.randn(1, 3, size, size)
    with torch.no_grad():
        with FlopCounterMode(display=False) as fc:
            model(x)
        macs = fc.get_total_flops() / 2
        torch.set_num_threads(1)
        for _ in range(5): model(x)
        t0 = time.time()
        for _ in range(runs): model(x)
    return macs, (time.time() - t0) / runs * 1000

rows = []
for name, mdl, size in [("MobileNetV3-S", make_student(False), IMG), ("ResNet-50", make_teacher(False), IMG),
                        ("ConvNet13 @32", ConvNet13(), 32), ("ConvNet13 @112", ConvNet13(), 112)]:
    macs, ms = macs_and_latency(mdl, size)
    rows.append(dict(model=name, params_M=round(n_params(mdl) / 1e6, 2), GMACs=round(macs / 1e9, 3), cpu_ms=round(ms, 1)))
print(pd.DataFrame(rows).to_string(index=False))

## Step 15. Putting the results into the Part 2 report

| Report item | Where it comes from |
|---|---|
| Table II, TBD settings | the `CFG` and `SV` cells (steps, lr, batch, `lam_c`, `tau`) and the Step 10 outputs |
| Table III (SVHN) | `svhn_table()` in Step 8 |
| Table IV (CeFA, ACER) | `table_cefa` / `latex_rows` in Step 13 |
| Figure 1 (curves) | `curves_*.pdf` saved in `ROOT`; caption must say **validation** error |
| Table V (efficiency) | Step 14 |
| Table VI (status) | what you actually finished |
| Software and hardware lines | the versions printed in Step 5 and the GPU name from Colab |

**Before you write the discussion, check:**

1. Does `audit_splits()` pass on the real manifest, and does `check_counts()` match Table I? If the counts differ, update the report's Table I rather than the other way round.
2. Did the smoke tests pass on the real data?
3. Is E3 better than E1 by more than the seed-to-seed spread? If not, say that plainly.
4. Is the EMA teacher better than the student (`acer` vs `acer_student`)?
5. Did you use any test-set result to choose a setting? If yes, redo that choice on the validation subjects.

**Report honestly:** the number of seeds you actually finished, every deviation from the paper (schedule, learning-rate schedule, batch split, BatchNorm handling), and any run that failed or was dropped.